## Step 1: Import Required Libraries

This step loads all libraries required for sentiment analysis, transformer
embeddings, aggregation, and dimensionality reduction.


In [43]:
# Core libraries
import os
import gc
import pandas as pd
import numpy as np

# NLP / Deep Learning
import torch
from tqdm import tqdm
from transformers import pipeline, AutoTokenizer, AutoModel

# Dimensionality Reduction
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD


## Step 2: Load Raw Review Datasets

These are the original expert and user review files.
No processing is applied at this stage.


In [44]:
# Load raw review data (test mode: 500 rows)
df_expert = pd.read_excel("../Metacritic dataset/ExpertReviews.xlsx").head(500)
df_user   = pd.read_excel("../Metacritic dataset/UserReviews.xlsx").head(500)


# Directory where processed outputs will be saved
PROCESSED_DIR = "../data/processed1"
os.makedirs(PROCESSED_DIR, exist_ok=True)


## Step 3: Text Cleaning Function

This function matches the original script exactly.
It ensures all review text is safe for Transformer models.


In [45]:
def clean_text(series):
    return (
        series.astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip(" '\"")
    )


## Step 4: Sentiment Analysis Function

Uses DistilBERT fine-tuned on SST-2.
Logic and parameters match the original script.


In [46]:
def run_sentiment(df, text_col="Rev"):
    device = 0 if torch.cuda.is_available() else -1
    print("Using GPU" if device == 0 else "Using CPU")

    sentiment = pipeline(
        "sentiment-analysis",
        model="distilbert-base-uncased-finetuned-sst-2-english",
        device=device,
        truncation=True,
        max_length=512,
        torch_dtype=torch.float16 if device == 0 else torch.float32
    )

    batch_size = 128 if device == 0 else 64
    outputs = []

    for i in tqdm(range(0, len(df), batch_size), desc="Sentiment"):
        outputs.extend(sentiment(df[text_col].iloc[i:i + batch_size].tolist()))

    df["sentiment_label"] = [o["label"] for o in outputs]
    df["sentiment_score"] = [
        o["score"] if o["label"] == "POSITIVE" else -o["score"]
        for o in outputs
    ]

    del sentiment, outputs
    gc.collect()
    torch.cuda.empty_cache()

    return df


## Step 5: Run Sentiment Analysis for Expert Reviews


In [47]:
expert_sentiment_path = f"{PROCESSED_DIR}/expert_sentiment.csv"

if not os.path.exists(expert_sentiment_path):

    df = df_expert.copy()
    df["Rev"] = clean_text(df["Rev"])
    print(f"Expert reviews: {len(df)}")

    df = run_sentiment(df)
    df.to_csv(expert_sentiment_path, index=False)

    print("✔ Expert sentiment saved")

else:
    print("✔ Expert sentiment already exists — skipping")


Expert reviews: 500
Using GPU


Device set to use cuda:0
Sentiment: 100%|██████████| 4/4 [00:08<00:00,  2.19s/it]

✔ Expert sentiment saved


## Step 6: Run Sentiment Analysis for User Reviews


In [48]:
user_sentiment_path = f"{PROCESSED_DIR}/user_sentiment.csv"

if not os.path.exists(user_sentiment_path):

    df = df_user.copy()
    df["Rev"] = clean_text(df["Rev"])
    print(f"User reviews: {len(df)}")

    df = run_sentiment(df)
    df.to_csv(user_sentiment_path, index=False)

    print("✔ User sentiment saved")

else:
    print("✔ User sentiment already exists — skipping")


User reviews: 500
Using GPU


Device set to use cuda:0
Sentiment: 100%|██████████| 4/4 [00:25<00:00,  6.45s/it]

✔ User sentiment saved


## Step 7: Transformer Embedding Function

Uses DistilBERT and mean pooling.
Logic is identical to the original script.


In [49]:
def run_embeddings(df, prefix):
    device = 0 if torch.cuda.is_available() else -1

    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    model = AutoModel.from_pretrained(
        "distilbert-base-uncased"
    ).to("cuda" if device == 0 else "cpu")
    model.eval()

    emb_batch = 64 if device == 0 else 32
    all_embeddings = []

    for i in tqdm(range(0, len(df), emb_batch), desc=f"{prefix} embeddings"):
        batch = df["Rev"].iloc[i:i + emb_batch].tolist()

        with torch.no_grad():
            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            ).to(model.device)

            emb = model(**encoded).last_hidden_state.mean(dim=1)
            all_embeddings.append(emb.cpu())

        torch.cuda.empty_cache()

    embeddings = torch.cat(all_embeddings).numpy()
    emb_dim = embeddings.shape[1]

    emb_cols = [f"{prefix}_emb_{i}" for i in range(emb_dim)]
    df = pd.concat([df.reset_index(drop=True),
                    pd.DataFrame(embeddings, columns=emb_cols)],
                   axis=1)

    del model, tokenizer, all_embeddings
    gc.collect()
    torch.cuda.empty_cache()

    return df, emb_cols


## Step 8: Generate Embeddings for Expert Reviews


In [50]:
expert_embeddings_path = f"{PROCESSED_DIR}/expert_embeddings_raw.csv"

if not os.path.exists(expert_embeddings_path):

    df = pd.read_csv(expert_sentiment_path)
    df["Rev"] = clean_text(df["Rev"])  # critical safety step

    df, expert_emb_cols = run_embeddings(df, prefix="expert")
    df.to_csv(expert_embeddings_path, index=False)

    print("✔ Expert embeddings generated")

else:
    print("✔ Expert embeddings already exist — skipping")


expert embeddings: 100%|██████████| 8/8 [00:02<00:00,  2.79it/s]


✔ Expert embeddings generated


## Step 9: Generate Embeddings for User Reviews


In [51]:
user_embeddings_path = f"{PROCESSED_DIR}/user_embeddings_raw.csv"

if not os.path.exists(user_embeddings_path):

    df = pd.read_csv(user_sentiment_path)
    df["Rev"] = clean_text(df["Rev"])

    df, user_emb_cols = run_embeddings(df, prefix="user")
    df.to_csv(user_embeddings_path, index=False)

    print("✔ User embeddings generated")

else:
    print("✔ User embeddings already exist — skipping")


user embeddings: 100%|██████████| 8/8 [00:20<00:00,  2.62s/it]


✔ User embeddings generated


## Step 10: Aggregation Function

Aggregates review-level features to the movie (URL) level.
This logic is identical to the original script.


In [52]:
def aggregate_features(df, emb_cols, prefix):
    return (
        df.groupby("url")
        .agg(
            **{
                f"{prefix}_sentiment_mean": ("sentiment_score", "mean"),
                f"{prefix}_positive_ratio": ("sentiment_label",
                                             lambda x: (x == "POSITIVE").mean()),
                f"{prefix}_review_count": ("sentiment_score", "count"),
                **{c: (c, "mean") for c in emb_cols}
            }
        )
        .reset_index()
    )


## Step 11: Aggregate Expert Features


In [53]:
expert_features_path = f"{PROCESSED_DIR}/expert_features_with_embeddings.csv"

if not os.path.exists(expert_features_path):

    df = pd.read_csv(expert_embeddings_path)
    emb_cols = [c for c in df.columns if c.startswith("expert_emb_")]

    expert_features = aggregate_features(df, emb_cols, "expert")
    expert_features.to_csv(expert_features_path, index=False)

    print("✔ Expert features aggregated")

else:
    print("✔ Expert aggregated features already exist — skipping")


✔ Expert features aggregated


## Step 12: Aggregate User Features


In [54]:
user_features_path = f"{PROCESSED_DIR}/user_features_with_embeddings.csv"

if not os.path.exists(user_features_path):

    df = pd.read_csv(user_embeddings_path)
    emb_cols = [c for c in df.columns if c.startswith("user_emb_")]

    user_features = aggregate_features(df, emb_cols, "user")
    user_features.to_csv(user_features_path, index=False)

    print("✔ User features aggregated")

else:
    print("✔ User aggregated features already exist — skipping")


✔ User features aggregated


## Step 13: Truncated SVD Function

Reduces embedding dimensionality while preserving variance.


In [59]:
def run_svd(df, emb_prefix, svd_prefix, n_components=50):

    emb_cols = [c for c in df.columns if c.startswith(emb_prefix)]
    meta_cols = [c for c in df.columns if c not in emb_cols]

    print(f"Embedding columns found: {len(emb_cols)}")

    X = df[emb_cols].values
    X_scaled = StandardScaler().fit_transform(X)

    # ✅ CRITICAL FIX: cap components for small datasets
    max_components = min(n_components, X_scaled.shape[0], X_scaled.shape[1])

    svd = TruncatedSVD(
        n_components=max_components,
        random_state=42
    )

    X_svd = svd.fit_transform(X_scaled)

    explained = svd.explained_variance_ratio_.sum()
    print(f"Explained variance retained: {explained:.2%}")
    print(f"SVD components used: {max_components}")

    svd_cols = [f"{svd_prefix}{i}" for i in range(max_components)]
    df_svd = pd.DataFrame(X_svd, columns=svd_cols)

    df_final = pd.concat(
        [
            df[meta_cols].reset_index(drop=True),
            df_svd.reset_index(drop=True)
        ],
        axis=1
    )

    return df_final


## Step 14: Apply SVD and Save Final Outputs


In [60]:
expert_svd_path = f"{PROCESSED_DIR}/expert_features_with_svd.csv"

if not os.path.exists(expert_svd_path):
    df = pd.read_csv(expert_features_path)
    df = run_svd(df, "expert_emb_", "expert_svd_")
    df.to_csv(expert_svd_path, index=False)
    print("✔ Expert SVD saved")
else:
    print("✔ Expert SVD already exists — skipping")


Embedding columns found: 768
Explained variance retained: 100.00%
SVD components used: 26
✔ Expert SVD saved


In [61]:
user_svd_path = f"{PROCESSED_DIR}/user_features_with_svd.csv"

if not os.path.exists(user_svd_path):
    df = pd.read_csv(user_features_path)
    df = run_svd(df, "user_emb_", "user_svd_")
    df.to_csv(user_svd_path, index=False)
    print("✔ User SVD saved")
else:
    print("✔ User SVD already exists — skipping")


Embedding columns found: 768
Explained variance retained: 100.00%
SVD components used: 28
✔ User SVD saved
